In [ ]:
import os
from dotenv import load_dotenv
from google import genai
import pandas as pd 
import nbformat 

### 1st Iteration: Converting a notebook into a dataframe

In [8]:
load_dotenv() # loading my env file
notebook_path = "/Users/ahmadwali04/Desktop/LeetCode/code.ipynb"
client = genai.Client()


def load_notebook_to_df(notebook_path):
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            nb=nbformat.read(f,as_version=4)
            rows = []
            for cell in nb.cells:
                if cell.cell_type in ['markdown','code']:
                    rows.append({
                        "chunk type": cell.cell_type,
                        "chunk" : cell.source.strip()
                    })
            df = pd.DataFrame(rows)
            return df
    except FileNotFoundError:
        print("File not found")
        return None 
    
def ask_about_chunk(df,index,question):
    selected_row = df.loc[index]
    chunk_type = selected_row["chunk type"]
    chunk_content = selected_row["chunk"]

    prompt = f"""
    You are an expert software engineer and LeetCode mentor. 
    Below is a specific content chunk extracted from my LeetCode Jupyter Notebook (Index: {index}).

    --- CHUNK TYPE: {chunk_type.upper()} ---
    {chunk_content}
    ---------------------------------

    USER QUESTION ABOUT THIS CHUNK:
    {question}
    """

    print(f"Sending Chunk #{index} to Gemini...")
        
    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )
        
        print("\n=== GEMINI'S ANALYSIS ===")
        print(response.text)
        print("=========================")
        
    except Exception as e:
        print(f"An error occurred while calling the Gemini API: {e}")


df = load_notebook_to_df(notebook_path)


# Display the dataframe (this will show the index, chunk type, and snippet)
if df is not None:
    # Set display options so pandas doesn't truncate the text too aggressively
    pd.set_option('display.max_colwidth', 100)
    print("Notebook parsed successfully! Here are your chunks:")
    display(df)

Notebook parsed successfully! Here are your chunks:


,chunk type,chunk
0,markdown,"# Contains Duplicate\nGiven an integer array nums, return true if any value appears more than on..."
1,code,"class Solution:\n def hasDuplicate(self, nums: List[int]) -> bool:\n x = set(nums)\n ..."
2,markdown,"# Valid Anagram\nGiven two strings s and t, return true if the two strings are anagrams of each ..."
3,code,"class Solution:\n def isAnagram(self, s: str, t: str) -> bool:\n if len(s) != len(t):\..."
4,markdown,"# Two Sum\n\nGiven an array of integers nums and an integer target, return the indices i and j s..."
5,code,"class Solution:\n def twoSum(self, nums: List[int], target: int) -> List[int]:\n seen ..."
6,markdown,"# Group Anagrams\nGiven an array of strings strs, group all anagrams together into sublists. You..."


In [14]:
my_question = "Explain what this problem is and how a solution might look"
ask_about_chunk(df, index=3, question=my_question)

Sending Chunk #3 to Gemini...

=== GEMINI'S ANALYSIS ===
As an expert software engineer and LeetCode mentor, let me break this down for you.

### 1. What is this problem?
This is the classic **"Valid Anagram"** problem (LeetCode #242).

**Definition:** An anagram is a word or phrase formed by rearranging the letters of a different word or phrase, typically using all the original letters exactly once.

**The core logic:** Two strings are anagrams if and only if they contain the **exact same characters with the exact same frequencies**.
*   *Example:* `"listen"` and `"silent"` are anagrams.
*   *Example:* `"rat"` and `"car"` are not anagrams.

---

### 2. How the solution works
Your provided code uses a **Frequency Counter (Hash Map)** approach. Here is the step-by-step breakdown of the logic:

#### Step A: Early Exit (Edge Case)
```python
if len(s) != len(t):
    return False
```
This is a constant-time $O(1)$ check. If the lengths differ, they cannot possibly be anagrams. Always look f

### 2nd Iteration: Converting chunks from notebook into vectors with SBERT

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def load_notebook_to_df_st(path):
    with open(path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    rows = []
    for cell_index, cell in enumerate(nb.cells):
        if cell.cell_type in ["markdown", "code"]:
            rows.append(
                {
                    "cell_index": cell_index,
                    "chunk type": cell.cell_type,
                    "chunk": cell.source.strip(),
                }
            )

    return pd.DataFrame(rows)


def build_embeddings_st(df):
    chunks = df["chunk"].fillna("").tolist()
    return st_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


def retrieve_chunks_st(df, embeddings, question, top_k=3):
    question_embedding = st_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    scores = np.dot(embeddings, question_embedding)
    top_indices = scores.argsort()[::-1][:top_k]
    results = df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results


def ask_rag_question_st(question, top_k=3):
    local_df = load_notebook_to_df_st(notebook_path)
    embeddings = build_embeddings_st(local_df)
    retrieved = retrieve_chunks_st(local_df, embeddings, question, top_k=top_k)

    context_parts = []
    for _, row in retrieved.iterrows():
        context_parts.append(
            f"[cell {row['cell_index']}] ({row['chunk type']}, score={row['score']:.3f})\n{row['chunk']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are an expert software engineer and LeetCode mentor.
Use only the notebook context below to answer the question.
If the context is insufficient, say what is missing and what you would need next.

NOTEBOOK CONTEXT:
{context}

USER QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
    )

    print(response.text)
    return retrieved


question = "Explain what this notebook chunk is doing and suggest a better approach."
retrieved = ask_rag_question_st(question, top_k=3)
display(retrieved)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9128.84it/s]



Based on the provided notebook context, here is the explanation and recommendation.

### Explanation of the Notebook Chunk
The notebook chunk (specifically [cell 2]) defines the **"Valid Anagram"** problem and captures your internal brainstorming process for solving it.

Your notes identify two primary strategies:
1.  **Frequency Hashmap:** Comparing the character counts of both strings. You correctly identified a pitfall: if you simply remove keys from a dictionary, you might incorrectly handle duplicate characters. The solution you landed on is to count the frequency of each character in both strings and compare the resulting maps.
2.  **Sorting:** Sorting both strings and comparing them for equality ($O(n \log n)$ time complexity).
3.  **Language-Specific Tip:** You noted that when using the `.get()` method in Python, providing a default value of `0` is a clean way to handle missing keys during the counting process.

### Suggested "Better" Approach

While your proposed Hashmap appro

,cell_index,chunk type,chunk,score
2,2,markdown,"# Valid Anagram\nGiven two strings s and t, return true if the two strings are anagrams of each ...",0.121209
8,8,markdown,,0.081059
6,6,markdown,,0.081059


### 3rd Iteration: Introduceing multiple documents & PDF's

In [ ]:
notebooks = "/Users/ahmadwali04/Desktop/personal/Research/IBM/QuantumML"
df2 = pd.DataFrame()
for file in os.listdir(notebooks):
    if file.endswith(".ipynb"):
        path = os.path.join(notebooks, file)
        temp = load_notebook_to_df(path)
        if temp is None:
            continue
        temp["notebook"] = file
        df2 = pd.concat([df2, temp], ignore_index=True)
display(df2)


,chunk type,chunk,notebook
0,markdown,---\ntitle: Introduction\ndescription: An introduction to quantum machine learning that sets exp...,introduction.ipynb
1,markdown,## Check-in questions\n\nWhat makes quantum states different from classical states?\n\n<Accordio...,introduction.ipynb
2,markdown,"## Course learning goals\n\nThrough completing this course, you can expect to build the followin...",introduction.ipynb
3,markdown,## Course structure\n\nThis course is made up of several lessons. Each lesson has several check-...,introduction.ipynb
4,markdown,"## Run your first QML code\n\nIt is often helpful to see where we're going, before breaking it d...",introduction.ipynb
...,...,...,...
225,markdown,"We can plot these first few optimization steps, although we would not expect any convergence aft...",qvc-qnn.ipynb
226,code,obj_func_vals_qc = objective_func_vals\n# import matplotlib.pyplot as plt\n\nplt.figure(figsize=...,qvc-qnn.ipynb
227,markdown,"### Closing\n\nTo recap, in this lesson we learned the workflow for binary classification of ima...",qvc-qnn.ipynb
228,markdown,## References\n\n\[1] [https://arxiv.org/abs/2405.00781](https://arxiv.org/abs/2405.00781),qvc-qnn.ipynb


As an observation, it does not read images within the jupyter notebook well. It also 

In [ ]:
query = "What is a quantum neural network? Please leave citations on which part of which notebook I should go read"
# query  = "What is the image in the 'Generalization' section of 'classical-ml-review.ipynb' trying to conveey?"

# Build embeddings for all chunks across notebooks
chunks = df2["chunk"].fillna("").tolist()
embeddings_all = st_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Retrieve top relevant chunks
query_embedding = st_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True,
)[0]

scores = np.dot(embeddings_all, query_embedding)
top_k = 12
top_indices = scores.argsort()[::-1][:top_k]

hits = df2.iloc[top_indices].copy()
hits["score"] = scores[top_indices]
hits = hits.sort_values("score", ascending=False)

# Build context with explicit citation handles
context_parts = []
for row_idx, row in hits.iterrows():
    context_parts.append(
        f"[CITATION: notebook={row['notebook']}, row={row_idx}, type={row['chunk type']}, score={row['score']:.3f}]\n"
        f"{str(row['chunk'])[:2000]}"
    )

context = "\n\n".join(context_parts)

prompt = f"""
You are an expert quantum ML tutor.
Answer the user's question using only the context below.
When you make a claim, add citations in this exact format:
[notebook: <name>, row: <row_index>]

If context is insufficient, say so clearly.

CONTEXT:
{context}

USER QUESTION:
{query}
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
)

print(response.text)
display(hits[["notebook", "chunk type", "score", "chunk"]])

The provided context does not contain a "Generalization" section in the `classical-ml-review.ipynb` notebook, nor does it provide information regarding images within that specific section. Therefore, the context is insufficient to answer your question.


,notebook,chunk type,score,chunk
186,qvc-qnn.ipynb,markdown,0.355541,We might try extending the optimization steps to make sure the optimizer didn't just get stuck i...
197,qvc-qnn.ipynb,code,0.339088,"pred_test = forward(circuit, np.array(test_images), res[""x""], estimator, observable)\n# pred_tes..."
183,qvc-qnn.ipynb,code,0.339088,"pred_test = forward(circuit, np.array(test_images), res[""x""], estimator, observable)\n# pred_tes..."
180,qvc-qnn.ipynb,markdown,0.316282,"## Qiskit Patterns Step 4: Post-process, return result in classical format\n\n### Testing and ac..."
181,qvc-qnn.ipynb,code,0.310675,import copy\nfrom sklearn.metrics import accuracy_score\nfrom qiskit.primitives import Statevect...
143,classical-ml-review.ipynb,markdown,0.309352,"## References\n\n\[1] [""Reinforcement Learning: An Introduction""](http://incompleteideas.net/boo..."
138,classical-ml-review.ipynb,markdown,0.303069,---\ntitle: Classical ML review\ndescription: A brief review of classical machine learning metho...
166,qvc-qnn.ipynb,markdown,0.298747,Let us also define a slightly different loss function that is a function of the variable paramet...
196,qvc-qnn.ipynb,code,0.287964,from sklearn.metrics import accuracy_score\nfrom qiskit.primitives import StatevectorEstimator a...
228,qvc-qnn.ipynb,markdown,0.287515,## References\n\n\[1] [https://arxiv.org/abs/2405.00781](https://arxiv.org/abs/2405.00781)


In [ ]:
temp = "/Users/ahmadwali04/Desktop/personal/Research/IBM/Temp"

from docling.document_converter import DocumentConverter

converter = DocumentConverter()


def load_pdf_to_df(notebook_path):

    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:

            nb=nbformat.read(f,as_version=4)
            rows = []
            for cell in nb.cells:
                if cell.cell_type in ['markdown','code']:
                    rows.append({
                        "chunk type": cell.cell_type,
                        "chunk" : cell.source.strip()
                    })
            df = pd.DataFrame(rows)
            return df
    except FileNotFoundError:
        print("File not found")
        return None 
 

df3 = pd.DataFrame()
for file in os.listdir(notebooks):
    if file.endswith(".ipynb" or file.endswith(".pdf")):
        path = os.path.join(notebooks, file)
        if file.endswith(".ipynb"):
            temp = load_notebook_to_df(path)
        if file.endswith(".pdf"):
            temp = load_pdf_to_df(path)
        if temp is None:
            continue
        temp["notebook"] = file
        df3 = pd.concat([df3, temp], ignore_index=True)
display(df3)




In [14]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

source = "1706.02275v4.pdf"  # or "https://arxiv.org/pdf/2408.09869"

result = converter.convert(source)

markdown_output = result.document.export_to_markdown()
print(markdown_output)

# 5. Optionally save the output to an actual .md file
with open("output.md", "w", encoding="utf-8") as f:
    f.write(markdown_output)


[INFO] 2026-07-16 15:56:00,237 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-16 15:56:00,240 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 15:56:00,242 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 15:56:00,240 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 15:56:00,242 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 15:56:01,563 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-16 15:56:01,563 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-16 15:56:02,400 [RapidOCR] download_file.py:95: Successfully saved to: /Users/ahmadwali04/Desktop/personal/Projects/NoteGraph/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-O

ImportError: 
RTDetrImageProcessor requires the Torchvision library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.

RTDetrImageProcessor requires the PIL library but it was not found in your environment. You can install it with pip:
`pip install pillow`. Please note that you may need to restart your runtime after installation.
